# Validate Silver geography and population

## Purpose

Validate the three Silver geography and population outputs using consistency, accuracy, completeness, auditability, validity, uniqueness, and timeliness checks. Results are persisted before the stop gate is enforced. One result row represents one check for one table and deterministic pipeline run. Known source limitations remain `FLAG`; unsafe grain, lineage, accounting, or total failures are `FAIL`.


## 1. Collect each small table once

The three Silver outputs and their two small Bronze sources are aggregated once into reusable metrics. This reports row preservation, controlled statuses, match coverage, official region coverage, lineage, and population reconciliation without scanning unrelated project or spatial sources.


In [ ]:
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

CREATE OR REPLACE TEMPORARY VIEW silver_geography_population_metrics AS
WITH psgc_source AS (
    SELECT COUNT(*) AS source_rows FROM `01-bronze`.psgc
),
psgc AS (
    SELECT
        COUNT(*) AS silver_rows,
        COUNT_IF(psgc_code IS NULL OR TRIM(psgc_code) = '') AS null_codes,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'psgc_code', psgc_code,
            'version', COALESCE(psgc_version, source_snapshot_id)
        )) AS duplicate_keys,
        COUNT_IF(place_type = 'UNASSIGNED') AS unassigned_levels,
        COUNT_IF(hierarchy_status <> 'VALID_HIERARCHY') AS hierarchy_flags,
        COUNT(*) - COUNT(DISTINCT place_name_standardized) AS repeated_standardized_names,
        COUNT_IF(source_snapshot_id IS NULL OR source_ingest_run_id IS NULL OR run_id IS NULL) AS missing_lineage,
        COUNT(DISTINCT source_snapshot_id) AS snapshot_count,
        MAX(source_snapshot_id) AS source_snapshot_id,
        MAX(transformation_rule_version) AS rule_version,
        MAX(run_id) AS pipeline_run_id
    FROM `02-silver`.silver_psgc_place
),
population_source AS (
    SELECT COUNT(*) AS source_rows FROM `01-bronze`.census_2024_table_c
),
reconciliation AS (
    SELECT
        COUNT(*) AS silver_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'source_file', source_file,
            'sheet_name', sheet_name,
            'source_row_number', source_row_number,
            'run_id', run_id
        )) AS duplicate_keys,
        COUNT_IF(match_status = 'MATCHED_EXACT_CONTEXT') AS exact_matches,
        COUNT_IF(match_status = 'MATCHED_ALIAS') AS alias_matches,
        COUNT_IF(match_status = 'AMBIGUOUS') AS ambiguous_rows,
        COUNT_IF(match_status = 'UNMATCHED') AS unmatched_rows,
        COUNT_IF(population_count IS NULL) AS invalid_population_rows,
        COUNT_IF(population_count IS NOT NULL AND population_count <= 0) AS non_positive_population_rows,
        COUNT_IF(row_disposition = 'ACCEPTED_PRIMARY') AS accepted_primary_rows,
        COUNT_IF(row_disposition = 'MATCHED_NON_PRIMARY_LEVEL') AS matched_non_primary_rows,
        COUNT_IF(row_disposition NOT IN (
            'ACCEPTED_PRIMARY', 'MATCHED_NON_PRIMARY_LEVEL', 'AMBIGUOUS',
            'UNMATCHED', 'INVALID_POPULATION', 'SOURCE_EXCEPTION'
        ) OR row_disposition IS NULL) AS invalid_dispositions,
        COUNT_IF(match_status NOT IN (
            'MATCHED_EXACT_CONTEXT', 'MATCHED_ALIAS', 'AMBIGUOUS', 'UNMATCHED', 'INVALID_SOURCE'
        ) OR match_status IS NULL) AS invalid_match_statuses,
        COUNT_IF(row_disposition = 'ACCEPTED_PRIMARY' AND (
            matched_place_type <> 'BARANGAY' OR population_count <= 0 OR matched_region_code IS NULL
        )) AS invalid_primary_rows,
        COUNT_IF(source_snapshot_id IS NULL OR source_ingest_run_id IS NULL
            OR psgc_source_snapshot_id IS NULL OR psgc_source_ingest_run_id IS NULL
            OR run_id IS NULL OR transformation_rule_version IS NULL) AS missing_lineage,
        COUNT(DISTINCT source_snapshot_id) AS source_snapshot_count,
        COUNT(DISTINCT psgc_source_snapshot_id) AS psgc_snapshot_count,
        COUNT(DISTINCT alias_version) AS alias_version_count,
        MAX(source_snapshot_id) AS source_snapshot_id,
        MAX(psgc_source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(run_id) AS pipeline_run_id,
        MAX(transformation_rule_version) AS rule_version,
        SUM(CASE WHEN row_disposition = 'ACCEPTED_PRIMARY' THEN population_count END) AS accepted_population
    FROM `02-silver`.silver_population_place_reconciliation
),
regions AS (
    SELECT
        COUNT(*) AS region_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'region', psgc_region_code,
            'year', reference_year,
            'source', source_name,
            'snapshot', source_snapshot_id,
            'psgc_snapshot', psgc_source_snapshot_id
        )) AS duplicate_keys,
        COUNT_IF(population_count IS NULL) AS missing_population_regions,
        COUNT_IF(reference_year <> 2024 OR population_reference_date <> DATE'2024-07-01') AS invalid_reference_dates,
        COUNT_IF(source_snapshot_id IS NULL OR source_ingest_run_id IS NULL
            OR psgc_source_snapshot_id IS NULL OR psgc_source_ingest_run_id IS NULL
            OR run_id IS NULL OR transformation_rule_version IS NULL) AS missing_lineage,
        SUM(source_row_count) AS contributing_source_rows,
        SUM(population_count) AS regional_population,
        MAX(source_snapshot_id) AS source_snapshot_id,
        MAX(run_id) AS pipeline_run_id,
        MAX(transformation_rule_version) AS rule_version
    FROM `02-silver`.silver_region_population
),
official_regions AS (
    SELECT COUNT(*) AS region_rows
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
),
unexpected_regions AS (
    SELECT COUNT(*) AS unexpected_rows
    FROM `02-silver`.silver_region_population AS population
    LEFT ANTI JOIN `02-silver`.silver_psgc_place AS place
        ON population.psgc_region_code = place.psgc_code
       AND population.psgc_source_snapshot_id = place.source_snapshot_id
       AND place.place_type = 'REGION'
),
invalid_alias_targets AS (
    SELECT COUNT(*) AS invalid_rows
    FROM `02-silver`.config_place_name_alias AS alias
    LEFT ANTI JOIN `02-silver`.silver_psgc_place AS place
        ON alias.canonical_psgc_code = place.psgc_code
       AND (alias.psgc_version IS NULL OR alias.psgc_version = place.psgc_version)
    WHERE alias.source_system = 'census_2024_table_c'
      AND alias.approval_status = 'APPROVED'
      AND alias.is_active = TRUE
)
SELECT
    psgc_source.source_rows AS source_rows,
    psgc.silver_rows AS silver_rows,
    psgc.null_codes,
    psgc.duplicate_keys,
    psgc.unassigned_levels,
    psgc.hierarchy_flags,
    psgc.repeated_standardized_names,
    psgc.missing_lineage,
    psgc.snapshot_count,
    psgc.source_snapshot_id,
    psgc.rule_version,
    psgc.pipeline_run_id,
    population_source.source_rows AS source_rows_1,
    reconciliation.silver_rows AS silver_rows_1,
    reconciliation.duplicate_keys AS duplicate_keys_1,
    reconciliation.exact_matches,
    reconciliation.alias_matches,
    reconciliation.ambiguous_rows,
    reconciliation.unmatched_rows,
    reconciliation.invalid_population_rows,
    reconciliation.non_positive_population_rows,
    reconciliation.accepted_primary_rows,
    reconciliation.matched_non_primary_rows,
    reconciliation.invalid_dispositions,
    reconciliation.invalid_match_statuses,
    reconciliation.invalid_primary_rows,
    reconciliation.missing_lineage AS missing_lineage_1,
    reconciliation.source_snapshot_count,
    reconciliation.psgc_snapshot_count,
    reconciliation.alias_version_count,
    reconciliation.source_snapshot_id AS source_snapshot_id_1,
    reconciliation.psgc_source_snapshot_id,
    reconciliation.pipeline_run_id AS pipeline_run_id_1,
    reconciliation.rule_version AS rule_version_1,
    reconciliation.accepted_population,
    regions.region_rows,
    regions.duplicate_keys AS duplicate_keys_2,
    regions.missing_population_regions,
    regions.invalid_reference_dates,
    regions.missing_lineage AS missing_lineage_2,
    regions.contributing_source_rows,
    regions.regional_population,
    regions.source_snapshot_id AS source_snapshot_id_2,
    regions.pipeline_run_id AS pipeline_run_id_2,
    regions.rule_version AS rule_version_2,
    official_regions.region_rows AS region_rows_1,
    unexpected_regions.unexpected_rows,
    invalid_alias_targets.invalid_rows
FROM psgc_source
CROSS JOIN psgc
CROSS JOIN population_source
CROSS JOIN reconciliation
CROSS JOIN regions
CROSS JOIN official_regions
CROSS JOIN unexpected_regions
CROSS JOIN invalid_alias_targets;


## 2. Build the validation result set

Stop checks protect source-row accounting, keys, lineage, controlled values, accepted barangay grain, alias references, official regions, and exact population reconciliation. Flag checks quantify unresolved matches, non-positive population, hierarchy exceptions, repeated names, and PSGC diagnostic differences. Coverage rows always retain a numerator, denominator, and percentage.


In [ ]:
CREATE OR REPLACE TEMPORARY VIEW silver_geography_population_checks AS
WITH m AS (SELECT * FROM silver_geography_population_metrics),
checks AS (
    SELECT 'silver_psgc_place' AS table_name, 'Completeness' AS attribute,
        'Bronze PSGC rows equal Silver PSGC rows' AS check_name,
        ABS(source_rows - silver_rows) AS failed_rows, source_rows AS total_rows, 'stop' AS action,
        CONCAT('Bronze=', source_rows, ', Silver=', silver_rows) AS details,
        source_snapshot_id, rule_version, pipeline_run_id
    FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Validity', 'PSGC code is present', null_codes, silver_rows, 'stop',
        CONCAT('Null or blank codes=', null_codes), source_snapshot_id, rule_version, pipeline_run_id FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Uniqueness', 'PSGC natural key is unique', duplicate_keys, silver_rows, 'stop',
        CONCAT('Duplicate natural keys=', duplicate_keys), source_snapshot_id, rule_version, pipeline_run_id FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Completeness', 'Unassigned geographic levels remain visible', unassigned_levels, silver_rows, 'flag',
        CONCAT('Unassigned level rows=', unassigned_levels), source_snapshot_id, rule_version, pipeline_run_id FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Consistency', 'Hierarchy exceptions remain visible', hierarchy_flags, silver_rows, 'flag',
        CONCAT('Hierarchy exception rows=', hierarchy_flags), source_snapshot_id, rule_version, pipeline_run_id FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Accuracy', 'Repeated standardized names require hierarchy context', repeated_standardized_names, silver_rows, 'flag',
        CONCAT('Rows beyond distinct standardized names=', repeated_standardized_names), source_snapshot_id, rule_version, pipeline_run_id FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Auditability', 'Required PSGC lineage is present', missing_lineage, silver_rows, 'stop',
        CONCAT('Rows missing lineage=', missing_lineage), source_snapshot_id, rule_version, pipeline_run_id FROM m
    UNION ALL SELECT 'silver_psgc_place', 'Timeliness', 'Exactly one PSGC snapshot is selected', ABS(snapshot_count - 1), 1, 'stop',
        CONCAT('Selected snapshot count=', snapshot_count), source_snapshot_id, rule_version, pipeline_run_id FROM m

    UNION ALL SELECT 'silver_population_place_reconciliation', 'Completeness', 'Every Table C row has one reconciliation result',
        ABS(source_rows_1 - silver_rows_1), source_rows_1, 'stop', CONCAT('Bronze=', source_rows_1, ', Silver=', silver_rows_1),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Uniqueness', 'Table C source-row grain is unique', duplicate_keys_1, silver_rows_1, 'stop',
        CONCAT('Duplicate source-row keys=', duplicate_keys_1), source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Consistency', 'Match statuses use the controlled contract', invalid_match_statuses, silver_rows_1, 'stop',
        CONCAT('Invalid match statuses=', invalid_match_statuses), source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Consistency', 'Every row has one controlled disposition', invalid_dispositions, silver_rows_1, 'stop',
        CONCAT('Invalid dispositions=', invalid_dispositions), source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Accuracy', 'Accepted primary rows are positive matched barangays', invalid_primary_rows, accepted_primary_rows, 'stop',
        CONCAT('Invalid accepted-primary rows=', invalid_primary_rows), source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Accuracy', 'Approved aliases reference the selected PSGC', invalid_rows,
        GREATEST(invalid_rows, 1), 'stop', CONCAT('Invalid approved alias targets=', invalid_rows),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Completeness', 'Exact contextual match coverage',
        silver_rows_1 - exact_matches, silver_rows_1, 'flag', CONCAT('Exact contextual matches=', exact_matches),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Completeness', 'Approved alias match coverage',
        alias_matches, silver_rows_1, 'flag', CONCAT('Approved alias matches=', alias_matches),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Completeness', 'Ambiguous rows remain visible',
        ambiguous_rows, silver_rows_1, 'flag', CONCAT('Ambiguous rows=', ambiguous_rows),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Completeness', 'Unmatched rows remain visible',
        unmatched_rows, silver_rows_1, 'flag', CONCAT('Unmatched rows=', unmatched_rows),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Validity', 'Population parsing exceptions remain visible',
        invalid_population_rows, silver_rows_1, 'flag', CONCAT('Unparseable population rows=', invalid_population_rows),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Validity', 'Non-positive population remains visible',
        non_positive_population_rows, silver_rows_1, 'flag', CONCAT('Non-positive population rows=', non_positive_population_rows),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Auditability', 'Required reconciliation lineage is present',
        missing_lineage_1, silver_rows_1, 'stop', CONCAT('Rows missing lineage=', missing_lineage_1),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m
    UNION ALL SELECT 'silver_population_place_reconciliation', 'Timeliness', 'One compatible source and PSGC snapshot are used',
        GREATEST(source_snapshot_count - 1, 0) + GREATEST(psgc_snapshot_count - 1, 0) + GREATEST(alias_version_count - 1, 0),
        3, 'stop', CONCAT('Table C snapshots=', source_snapshot_count, ', PSGC snapshots=', psgc_snapshot_count, ', alias versions=', alias_version_count),
        source_snapshot_id_1, rule_version_1, pipeline_run_id_1 FROM m

    UNION ALL SELECT 'silver_region_population', 'Completeness', 'All official PSGC regions are represented',
        ABS(region_rows_1 - region_rows), region_rows_1, 'stop', CONCAT('Official regions=', region_rows_1, ', output regions=', region_rows),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Consistency', 'No unexpected region code is published',
        unexpected_rows, region_rows, 'stop', CONCAT('Unexpected regions=', unexpected_rows),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Uniqueness', 'Region population natural key is unique',
        duplicate_keys_2, region_rows, 'stop', CONCAT('Duplicate region keys=', duplicate_keys_2),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Completeness', 'Missing official-region population remains visible',
        missing_population_regions, region_rows, 'flag', CONCAT('Regions missing population=', missing_population_regions),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Validity', 'Population reference date is July 1, 2024',
        invalid_reference_dates, region_rows, 'stop', CONCAT('Rows with invalid reference date=', invalid_reference_dates),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Auditability', 'Required region population lineage is present',
        missing_lineage_2, region_rows, 'stop', CONCAT('Rows missing lineage=', missing_lineage_2),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Consistency', 'Contributing row count reconciles to accepted primary rows',
        ABS(contributing_source_rows - accepted_primary_rows), accepted_primary_rows, 'stop',
        CONCAT('Accepted rows=', accepted_primary_rows, ', regional contributors=', contributing_source_rows),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
    UNION ALL SELECT 'silver_region_population', 'Accuracy', 'Regional population reconciles to accepted Table C population',
        ABS(regional_population - accepted_population), accepted_primary_rows, 'stop',
        CONCAT('Accepted population=', accepted_population, ', regional population=', regional_population),
        source_snapshot_id_2, rule_version_2, pipeline_run_id_2 FROM m
)
SELECT
    SHA2(CONCAT_WS('|', pipeline_run_id, 'silver-geography-population-validation-v1'), 256) AS validation_run_id,
    pipeline_run_id,
    'silver' AS layer,
    table_name,
    source_snapshot_id,
    rule_version,
    attribute AS data_quality_attribute,
    check_name AS data_quality_check,
    CAST(failed_rows AS BIGINT) AS failed_rows,
    CAST(total_rows AS BIGINT) AS total_rows,
    CASE WHEN total_rows > 0 THEN ROUND(100.0 * failed_rows / total_rows, 4) END AS percentage,
    CASE WHEN failed_rows = 0 THEN 'PASS' WHEN action = 'stop' THEN 'FAIL' ELSE 'FLAG' END AS status,
    action,
    details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks;


## 3. Persist evidence, then enforce the stop gate

The result table is separate from the fixed Bronze and CONFIG writers. A deterministic merge makes an exact rerun safe without duplicating evidence. All checks are stored before `ASSERT_TRUE` blocks unsafe downstream use. Flags remain queryable for review.


In [ ]:
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL,
    pipeline_run_id STRING NOT NULL,
    layer STRING NOT NULL,
    table_name STRING NOT NULL,
    source_snapshot_id STRING,
    rule_version STRING,
    data_quality_attribute STRING NOT NULL,
    data_quality_check STRING NOT NULL,
    failed_rows BIGINT,
    total_rows BIGINT,
    percentage DOUBLE,
    status STRING NOT NULL,
    action STRING NOT NULL,
    details STRING,
    run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING silver_geography_population_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Silver geography and population checks failed'
    )
)
FROM silver_geography_population_checks;


# Summary

Validation writes reusable evidence to `04-validation.silver_dq_results` and blocks downstream use when a stop check fails. The checks cover all seven applicable quality attributes, exact source-row accounting, matching coverage, population validity, official region coverage, snapshot compatibility, lineage, natural keys, and total reconciliation. No source rows or flags are deleted. Requires Databricks execution. The next dependency is human review of unresolved matches, followed later by Gold region and population tables.


In [ ]:
SELECT
    validation_run_id,
    table_name,
    data_quality_attribute,
    status,
    action,
    failed_rows,
    total_rows,
    percentage,
    data_quality_check,
    details
FROM silver_geography_population_checks
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;
